<a href="https://colab.research.google.com/github/rbvwolf/volt-guardian/blob/main/volt_guardian.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Eğer SHAP Colab'de yüklü değilse, bu satır onu Google sunucusuna anında kurar:
!pip install shap -q

# Alet Çantamızı (Kütüphaneleri) Çağırıyoruz:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import xgboost as xgb
import shap

print("Tüm sistemler aktif! Volt-Guardian XAI projesi kodlamaya hazır.")

Tüm sistemler aktif! Volt-Guardian XAI projesi kodlamaya hazır.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
dosya_yolu = '/content/drive/MyDrive/Volt-Guardian/ev_battery_degradation_v1.csv'
df = pd.read_csv(dosya_yolu)

display(df.head())

print("data set")
df.info()

,Vehicle_ID,Car_Model,Battery_Type,Battery_Capacity_kWh,Vehicle_Age_Months,Total_Charging_Cycles,Avg_Temperature_C,Fast_Charge_Ratio,Avg_Discharge_Rate_C,Driving_Style,Internal_Resistance_Ohm,SoH_Percent,Battery_Status
0,1fb46ae8,Tesla Model 3,NMC,75.0,41,390,21.5,0.51,2.22,Aggressive,0.0362,94.60,Healthy
1,b7ef35aa,Tesla Model 3,NMC,75.0,29,401,18.0,0.62,1.34,Aggressive,0.0333,95.68,Healthy
2,76cb49e0,Ford Mustang Mach-E,NMC,88.0,71,941,18.4,0.78,1.48,Conservative,0.0526,89.80,Healthy
3,456a7aef,Ford Mustang Mach-E,NMC,88.0,57,378,10.8,0.61,0.72,Moderate,0.0314,96.29,Healthy
4,bd758049,Tesla Model 3,NMC,75.0,58,239,30.3,0.89,1.48,Conservative,0.0297,96.75,Healthy


data set
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Vehicle_ID               10000 non-null  object 
 1   Car_Model                10000 non-null  object 
 2   Battery_Type             10000 non-null  object 
 3   Battery_Capacity_kWh     10000 non-null  float64
 4   Vehicle_Age_Months       10000 non-null  int64  
 5   Total_Charging_Cycles    10000 non-null  int64  
 6   Avg_Temperature_C        10000 non-null  float64
 7   Fast_Charge_Ratio        10000 non-null  float64
 8   Avg_Discharge_Rate_C     10000 non-null  float64
 9   Driving_Style            10000 non-null  object 
 10  Internal_Resistance_Ohm  10000 non-null  float64
 11  SoH_Percent              10000 non-null  float64
 12  Battery_Status           10000 non-null  object 
dtypes: float64(6), int64(2), object(5)
memory usage: 1015.8+ KB


In [7]:
# 1. Çöp kolonları atıyoruz (ID'nin batarya ömrüne etkisi yoktur)
df_temiz = df.drop(columns=['Vehicle_ID'])

# 2. Metin (object) olan değişkenleri 0 ve 1'lerden oluşan sayılara çeviriyoruz
# pd.get_dummies bu işi bizim için otomatik yapar
df_islenmis = pd.get_dummies(df_temiz, columns=['Car_Model', 'Battery_Type', 'Driving_Style'], drop_first=True)

# Yeni verinin röntgenini çekelim (Artık hiç 'object' kalmamış olması lazım)
print("\n--- İşlenmiş Veri Seti Özeti ---")
df_islenmis.info()

# İlk 5 satırın yeni haline bakalım
display(df_islenmis.head())


--- İşlenmiş Veri Seti Özeti ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 16 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Battery_Capacity_kWh           10000 non-null  float64
 1   Vehicle_Age_Months             10000 non-null  int64  
 2   Total_Charging_Cycles          10000 non-null  int64  
 3   Avg_Temperature_C              10000 non-null  float64
 4   Fast_Charge_Ratio              10000 non-null  float64
 5   Avg_Discharge_Rate_C           10000 non-null  float64
 6   Internal_Resistance_Ohm        10000 non-null  float64
 7   SoH_Percent                    10000 non-null  float64
 8   Battery_Status                 10000 non-null  object 
 9   Car_Model_Ford Mustang Mach-E  10000 non-null  bool   
 10  Car_Model_Hyundai Ioniq 5      10000 non-null  bool   
 11  Car_Model_Tesla Model 3        10000 non-null  bool   
 12  Car_Model_Wul

,Battery_Capacity_kWh,Vehicle_Age_Months,Total_Charging_Cycles,Avg_Temperature_C,Fast_Charge_Ratio,Avg_Discharge_Rate_C,Internal_Resistance_Ohm,SoH_Percent,Battery_Status,Car_Model_Ford Mustang Mach-E,Car_Model_Hyundai Ioniq 5,Car_Model_Tesla Model 3,Car_Model_Wuling Air EV,Battery_Type_NMC,Driving_Style_Conservative,Driving_Style_Moderate
0,75.0,41,390,21.5,0.51,2.22,0.0362,94.60,Healthy,False,False,True,False,True,False,False
1,75.0,29,401,18.0,0.62,1.34,0.0333,95.68,Healthy,False,False,True,False,True,False,False
2,88.0,71,941,18.4,0.78,1.48,0.0526,89.80,Healthy,True,False,False,False,True,True,False
3,88.0,57,378,10.8,0.61,0.72,0.0314,96.29,Healthy,True,False,False,False,True,False,True
4,75.0,58,239,30.3,0.89,1.48,0.0297,96.75,Healthy,False,False,True,False,True,True,False


In [ ]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive', force_remount=True)